# HAKE-MER — Step 0 baseline (GoEmotions protocol)

Definitive **flat PLM** runs aligned with Demszky et al. (GoEmotions): **batch 16**, **LR 5e-5**, **4 epochs**, best **validation F1-macro** checkpoint, seeds 42 / 123 / 456.

Trains **DistilBERT-base** then **RoBERTa-base** (same optimization contract).

### Before you start

1. **Runtime → Factory reset runtime** (clean rerun).
2. **Runtime → Change runtime type → GPU**.
3. **Runtime → Run all** (~1.5–3 h for both backbones on a T4).
4. Download the zip from the last cell.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Use Runtime → Change runtime type → GPU, then run this cell again."
    )
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import subprocess
from getpass import getpass
from pathlib import Path

REPO = "khalef-khalil/marii"
WORKDIR = Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"


def git_token() -> str:
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return getpass("GitHub token (repo read), if clone fails: ")


def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(
            ["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"],
            check=True,
        )
        return
    r = subprocess.run(
        ["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)],
        capture_output=True,
    )
    if r.returncode == 0 and (WORKDIR / "run_baseline_campaign.sh").is_file():
        return
    token = git_token()
    if WORKDIR.is_dir():
        subprocess.run(["rm", "-rf", str(WORKDIR)], check=True)
    authed = f"https://{token}@github.com/{REPO}.git"
    subprocess.run(["git", "clone", "--depth", "1", authed, str(WORKDIR)], check=True)


clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

In [ ]:
!pip install -q -r requirements-train.txt

In [ ]:
!./run_baseline_campaign.sh --backbone distilbert-base-uncased --epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0

In [ ]:
!./run_baseline_campaign.sh --backbone roberta-base --epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0

In [ ]:
import json
from pathlib import Path

expected = {
    "baseline_plm_distilbert_base_uncased_campaign.json",
    "baseline_plm_roberta_base_campaign.json",
}
found = {p.name for p in Path("reference/artifacts").glob("baseline_plm_*_campaign.json")}
missing = expected - found
if missing:
    raise FileNotFoundError(f"Missing campaign JSON: {missing}. Run both training cells.")

for path in sorted(Path("reference/artifacts").glob("baseline_plm_*_campaign.json")):
    campaign = json.loads(path.read_text(encoding="utf-8"))
    proto = campaign.get("protocol", {})
    epochs_ran = len(campaign["runs"][0]["history"])
    print(f"\n=== {path.name} ===")
    print(f"  protocol: {proto}")
    print(f"  epochs in history: {epochs_ran}")
    if proto.get("epochs", 4) != 4 or epochs_ran < 4:
        raise ValueError("Wrong protocol: need 4 epochs and fresh git main. Factory reset and Run all.")
    for name, block in campaign["test_aggregate"].items():
        print(f"  {name}: {block['mean']:.4f} ± {block['std']:.4f}")

In [ ]:
import zipfile
from google.colab import files

zip_path = Path("/content/baseline_plm_step0_goemotions_protocol.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for campaign in sorted(Path("reference/artifacts").glob("baseline_plm_*_campaign.json")):
        zf.write(campaign, campaign.name)
    for metrics_file in sorted(Path("runs").glob("*_baseline_plm/metrics.json")):
        zf.write(metrics_file, f"{metrics_file.parent.name}/{metrics_file.name}")

print(f"Zip: {zip_path.name} ({zip_path.stat().st_size / 1e3:.1f} KB)")
files.download(str(zip_path))